Rapidly-exploring Random Trees (RRT)
====================================

**Topic**: Introduction to RRT planners

Version | Author
------------ | -------------
0.3 | Björn Hein


This worksheet **explains the basic concepts of path planning using an RRT planner**.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-4-0-Test-Environments (collision checker, benchmarks) and IP-5-0-PRM_Basics (probabilistic roadmaps)

**Dependencies**:

* IPRRT.py (the planners RRTSimple, RRT and RRTEdges), IPVISRRT.py (visualization), IPTestSuite.py (benchmarks) and IPVISRRTBasics.py (figures of this notebook)

Important links are:

* General Info: https://www.python.org
* NetworkX: https://networkx.org
* NumPy: https://numpy.org/doc/stable/
* Matplotlib: https://matplotlib.org/stable/

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Idea

A PRM (see IP-5-0) builds a **roadmap** for the whole free space in a learning phase and can answer many queries afterwards (**multi-query**). Often only **one** path from a start $q_s$ to a goal $q_g$ is needed. Then it is sufficient to explore the free space only as far as necessary (**single-query**):

* An RRT grows a **tree** rooted at the start $q_s$.
* Random samples pull the tree into the unexplored regions of the free space; it grows "rapidly" into the free space (see "Why rapidly-exploring?" below).
* From time to time it is tested, whether the goal can be connected to the tree. If so, the path is found: in a tree there is exactly one path between two nodes.
* There is no separate query phase. Like the PRM the RRT works in the implicit C-space: it only asks the collision checker.

RRTs are **probabilistically complete** (if a path exists, the probability to find it approaches 1 with an increasing number of nodes), but the paths are **not optimal**.

## Basic RRT algorithm

Tree: $T = (V, E)$, parameters: number of nodes $N$, goal test every $k$ nodes

1. $V \leftarrow \{q_s\}$ and $E \leftarrow \emptyset$
2. Loop until $|V| = N$

   1. **every $k$ nodes (goal test)**: $q_{near} \leftarrow$ nearest node of $T$ to $q_g$; **if** line between $q_{near}$ and $q_g$ is collision free **then** add $q_g$ to $T$ and **return** the path from $q_s$ to $q_g$
   2. $q_{rand} \leftarrow$ a **randomly** chosen **free configuration**
   3. $q_{near} \leftarrow$ the **nearest node** of $T$ to $q_{rand}$
   4. $q_{new} \leftarrow$ EXTEND($q_{near}$, $q_{rand}$)
   5. **if** line between $q_{near}$ and $q_{new}$ is collision free **then** $V \leftarrow V \cup \{q_{new}\}$ and $E \leftarrow E \cup \{(q_{near}, q_{new})\}$

3. **return** failure (no path found)

EXTEND decides how far the tree grows towards $q_{rand}$:

* `RRTSimple` (IPRRT.py): the whole way, $q_{new} = q_{rand}$
* `RRT` (IPRRT.py): half the way, $q_{new} = q_{near} + 0.5 \cdot (q_{rand} - q_{near})$
* other strategies: a fixed step size $\epsilon$, or - as discussed in the lecture - control laws modifying the direction of the new connections (e.g. for non-holonomic robots)

Step 3 searches the nearest **node**. The slides (RDT) search the nearest **point of the tree**, which may lie inside an edge: `RRTEdges` (IPRRT.py), see the variant below.

In IPRRT.py the parameters are called *numberOfGeneratedNodes* ($N$) and *testGoalAfterNumberOfNodes* ($k$).

### Example

The following figures show `RRTSimple` on the scene of IP-5-0 ($N = 200$, $k = 10$). Every step is recorded (see `IPVISRRTBasics.py`); the algorithm is the same as in IPRRT.py and in the implementation below.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from IPEnvironment import CollisionChecker
import IPVISRRTBasics as vis

scene, limits = vis.lectureScene()
exampleChecker = CollisionChecker(scene, limits=limits)
exampleStart, exampleGoal = [1.5, 3.0], [21.0, 9.5]

# RRT with recording of every step: the tree, the list of events and the path
exampleTree, exampleEvents, examplePath = vis.traceRRT(exampleChecker, exampleStart, exampleGoal,
                                                       numberOfGeneratedNodes=200, testGoalAfterNumberOfNodes=10,
                                                       extendFraction=1.0, seed=16)
print(f"{len(exampleEvents)} steps, {exampleTree.number_of_nodes()} nodes, path: {examplePath}")

At the beginning the tree contains only the start. Therefore every random configuration $q_{rand}$ is connected to $q_s$ (steps B to E):

In [ ]:
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, exampleEvents, "first", legend=True)

Later $q_{near}$ is the node of the tree nearest to $q_{rand}$, not necessarily the start. The tree grows into all directions:

In [ ]:
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, exampleEvents, "added")

If the line between $q_{near}$ and $q_{new}$ is in collision, $q_{new}$ is discarded (step E). It is **not** tried to connect it to another node of the tree:

In [ ]:
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, exampleEvents, "collision")

Every $k = 10$ nodes it is tested, whether the goal can be connected to the nearest node of the tree (step A):

In [ ]:
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, exampleEvents, "goalCollision")

As soon as this connection is collision-free, the planning is finished. The path is the (unique) path in the tree from $q_s$ to $q_g$:

In [ ]:
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, exampleEvents, "goalConnected")

**Try it** (optional, needs `ipywidgets`): step through the whole run.

In [ ]:
from ipywidgets import interact, IntSlider

interact(lambda step: vis.plotStep(exampleChecker, exampleStart, exampleGoal, exampleEvents, step),
         step=IntSlider(min=0, max=len(exampleEvents) - 1, value=0));

## Why "rapidly-exploring"?

Which node of the tree is extended, depends on $q_{rand}$: it is the nearest node. The region of all configurations, which are nearer to a certain node than to all other nodes, is the **Voronoi cell** of this node. So the probability to extend a node is the share of its Voronoi cell in the free space.

Nodes at the border of the explored region have large Voronoi cells, nodes inside the tree only small ones. Therefore the tree mostly grows at its frontier into the unexplored space (**Voronoi bias**). The figures show a tree (without goal test) after 5, 20 and 80 nodes: the darker a cell, the larger it is. The percentage is the probability, that the node with the largest cell is extended next.

In [ ]:
growingTree, _, _ = vis.traceRRT(exampleChecker, exampleStart, exampleGoal, numberOfGeneratedNodes=81,
                                 testGoalAfterNumberOfNodes=10**6, seed=16)
vis.showVoronoiBias(exampleChecker, exampleStart, growingTree, sizes=(5, 20, 80))

## Variant: extend only part of the way

The class `RRT` in IPRRT.py extends the tree only half the way towards $q_{rand}$ (dotted: the part, which is not extended):

In [ ]:
halfTree, halfEvents, halfPath = vis.traceRRT(exampleChecker, exampleStart, exampleGoal,
                                              numberOfGeneratedNodes=200, testGoalAfterNumberOfNodes=10,
                                              extendFraction=0.5, seed=16)
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, halfEvents, "added", legend=True)

Both variants with the same random numbers. The shorter edges lead to a denser tree; whether this needs more or less nodes depends on the scene (see the statistics below).

In [ ]:
vis.showVariants(exampleChecker, exampleStart, exampleGoal, [
    (f"RRTSimple: {exampleTree.number_of_nodes()} nodes, path length {vis.pathLength(exampleTree, examplePath):.1f}",
     exampleTree, examplePath),
    (f"RRT (half the way): {halfTree.number_of_nodes()} nodes, path length {vis.pathLength(halfTree, halfPath):.1f}",
     halfTree, halfPath),
])

## Variant: nearest point of the tree (nodes and edges)

On the slides (RDT) the nearest point is searched in $S(G)$, the set of all points reached by the tree: $q_n \leftarrow$ nearest($S(G)$, $\alpha(i)$), "not restricted to points" - i.e. not restricted to the nodes. The nearest point may lie **inside an edge**:

* The nearest point of an edge from $a$ to $b$ to a configuration $q$ is the projection of $q$ onto the edge: $t = \frac{(q - a) \cdot (b - a)}{|b - a|^2}$, limited to $[0, 1]$, and $q_{near} = a + t \, (b - a)$. For $t = 0$ or $t = 1$ it is the node $a$ or $b$.
* Searching all edges gives the nearest point of the tree.
* If $q_{near}$ lies inside an edge and the new connection is collision-free, $q_{near}$ is inserted as a new node: the edge is **split** into two edges.

The class `RRTEdges` in IPRRT.py is `RRTSimple` with this nearest point (function `nearestPointOfTree`). *numberOfGeneratedNodes* counts only the start and the added random configurations, the nodes inserted into edges come on top.

In [ ]:
import numpy as np

def nearestPointOnEdge(a, b, q):
    # nearest point of the edge from a to b to q: projection onto the edge, t limited to [0, 1]
    a, b, q = np.array(a, dtype=float), np.array(b, dtype=float), np.array(q, dtype=float)
    t = np.clip(np.dot(q - a, b - a) / np.dot(b - a, b - a), 0.0, 1.0)
    return a + t * (b - a), t

print(nearestPointOnEdge([0, 0], [4, 0], [1, 2]))    # inside the edge: t = 0.25
print(nearestPointOnEdge([0, 0], [4, 0], [-1, 2]))   # before the edge: the node a, t = 0

The example from above with `RRTEdges` (same random numbers). As soon as the tree has edges, the nearest point often lies inside an edge (dashed: the connection to the nearest node, which `RRTSimple` would use):

In [ ]:
edgesTree, edgesEvents, edgesPath = vis.traceRRT(exampleChecker, exampleStart, exampleGoal,
                                                 numberOfGeneratedNodes=200, testGoalAfterNumberOfNodes=10,
                                                 seed=16, nearestPoint=True)
vis.plotSituation(exampleChecker, exampleStart, exampleGoal, edgesEvents, "split", legend=True)

The new edge is perpendicular to the split edge. Comparison of the final trees; the recorded run is the same as the one of `RRTEdges` from IPRRT.py:

In [ ]:
import random
from IPRRT import RRTEdges

random.seed(16)
rrtEdges = RRTEdges(exampleChecker)
edgesSolution = rrtEdges.planPath([exampleStart], [exampleGoal],
                                  {"numberOfGeneratedNodes": 200, "testGoalAfterNumberOfNodes": 10})
moduleEdges = {frozenset(0 if node == "start" else node for node in edge) for edge in rrtEdges.graph.edges()}
print("same tree as RRTEdges:", moduleEdges == set(map(frozenset, edgesTree.edges())))

vis.showVariants(exampleChecker, exampleStart, exampleGoal, [
    (f"RRTSimple: {exampleTree.number_of_nodes()} nodes, path length {vis.pathLength(exampleTree, examplePath):.1f}",
     exampleTree, examplePath),
    (f"RRTEdges: {edgesTree.number_of_nodes()} nodes ({len(rrtEdges.splitNodes)} inserted into edges), "
     f"path length {vis.pathLength(edgesTree, edgesPath):.1f}", edgesTree, edgesPath),
])

### Why the nearest point?

Without obstacles the difference is easier to see (as in the figures on the slides): both trees with the same 100 random configurations in $[0, 1]^2$, starting in the middle.

In [ ]:
freeChecker = CollisionChecker(dict(), limits=[[0.0, 1.0], [0.0, 1.0]])
freeTrees = []
for name, nearestPoint in [("nearest node (RRTSimple)", False), ("nearest point (RRTEdges)", True)]:
    tree, _, _ = vis.traceRRT(freeChecker, [0.5, 0.5], None, numberOfGeneratedNodes=100,
                              testGoalAfterNumberOfNodes=10**6, seed=3, nearestPoint=nearestPoint)
    freeTrees.append((f"{name}: {tree.number_of_nodes()} nodes, total edge length {vis.treeLength(tree):.2f}",
                      tree, []))
vis.showVariants(freeChecker, [0.5, 0.5], None, freeTrees, panelWidth=5.0)

Observations:

* With the nearest node, new edges often start far away from the tree's nearest part: long edges run past other branches and even cross them.
* With the nearest point, every new edge is the shortest connection to the tree. It is perpendicular to the split edge and never crosses an existing edge (a crossing point would be nearer to the new configuration). The tree branches like the figures on the slides, its total edge length is about a sixth smaller.
* The price: for every new configuration all edges are searched (the k-d tree of `RRTSimple` only finds nodes), and the inserted nodes make the tree larger.

# Implementation

Following a compact implementation of the basic RRT. It uses the same steps (A to E) as the algorithm above and as the classes in IPRRT.py.

In [ ]:
import random
import math
import networkx as nx
import numpy as np

The random free configuration is generated as in IP-5-0. The nearest node is searched by brute force (IPRRT.py uses a k-d tree from `scipy.spatial`, which is much faster for large trees).

In [ ]:
def getRandomFreePosition(collChecker):
    limits = collChecker.getEnvironmentLimits()
    pos = [random.uniform(limit[0], limit[1]) for limit in limits]
    while collChecker.pointInCollision(pos):
        pos = [random.uniform(limit[0], limit[1]) for limit in limits]
    return pos


def nearestNode(tree, pos):
    """Node of the tree nearest to pos (brute force)"""
    return min(tree.nodes, key=lambda node: math.dist(tree.nodes[node]['pos'], pos))


def extend(qNear, qRand, fraction=1.0):
    """New configuration on the way from qNear to qRand: fraction 1.0 = qRand itself, 0.5 = half the way"""
    if fraction == 1.0:
        return qRand
    return list(fraction * (np.array(qRand) - np.array(qNear)) + np.array(qNear))

In [ ]:
def planRRT(collisionChecker, start, goal, numberOfGeneratedNodes, testGoalAfterNumberOfNodes, fraction=1.0):
    """Basic RRT. Returns the tree and the path (list of nodes, empty if no path was found)."""
    tree = nx.Graph()
    tree.add_node(0, pos=start)   # node 0 is the start

    while tree.number_of_nodes() < numberOfGeneratedNodes:
        # A. goal test every testGoalAfterNumberOfNodes nodes
        if tree.number_of_nodes() % testGoalAfterNumberOfNodes == 0:
            qNear = nearestNode(tree, goal)
            if not collisionChecker.lineInCollision(tree.nodes[qNear]['pos'], goal):
                tree.add_node("goal", pos=goal)
                tree.add_edge(qNear, "goal")
                return tree, nx.shortest_path(tree, 0, "goal")

        # B. random free configuration
        qRand = getRandomFreePosition(collisionChecker)
        # C. nearest node of the tree
        qNear = nearestNode(tree, qRand)
        # D. extend the tree towards qRand
        qNew = extend(tree.nodes[qNear]['pos'], qRand, fraction)
        # E. add the new node, if the connection is collision-free
        if not collisionChecker.lineInCollision(tree.nodes[qNear]['pos'], qNew):
            newNode = tree.number_of_nodes()
            tree.add_node(newNode, pos=qNew)
            tree.add_edge(qNear, newNode)

    return tree, []

Test on the example scene: with the same seed of the random generator, the implementation generates exactly the tree of the example above, and so does `RRTSimple` from IPRRT.py.

In [ ]:
from IPRRT import RRTSimple, RRT
from IPVISRRT import rrtPRMVisualize

random.seed(16)
tree, path = planRRT(exampleChecker, exampleStart, exampleGoal, numberOfGeneratedNodes=200, testGoalAfterNumberOfNodes=10)
print("path:", path)
print("same tree as in the example:", set(map(frozenset, tree.edges())) == set(map(frozenset, exampleTree.edges())))

random.seed(16)
rrtSimple = RRTSimple(exampleChecker)
solution = rrtSimple.planPath([exampleStart], [exampleGoal],
                              {"numberOfGeneratedNodes": 200, "testGoalAfterNumberOfNodes": 10})
moduleEdges = {frozenset(0 if node == "start" else node for node in edge) for edge in rrtSimple.graph.edges()}
print("same tree as RRTSimple:", moduleEdges == set(map(frozenset, tree.edges())))

fig, ax = plt.subplots(figsize=(8, 4.5))
rrtPRMVisualize(rrtSimple, solution, ax=ax, nodeSize=40)
vis.setupAxes(ax, exampleChecker, "RRTSimple from IPRRT.py, visualized with rrtPRMVisualize")

# Benchmarks

The three planners of IPRRT.py on the benchmarks of `IPTestSuite.py` (with the parameters of the previous version of this notebook). The results change with every run of the cell.

In [ ]:
import IPTestSuite as ts

rrtConfig = dict()
rrtConfig["numberOfGeneratedNodes"] = 100
rrtConfig["testGoalAfterNumberOfNodes"] = 10

fig, axes = plt.subplots(3, len(ts.benchList), figsize=(4.2 * len(ts.benchList), 14.0))
for row, plannerClass in enumerate([RRTSimple, RRT, RRTEdges]):
    for ax, benchmark in zip(axes[row], ts.benchList):
        planner = plannerClass(benchmark.collisionChecker)
        solution = planner.planPath(benchmark.startList, benchmark.goalList, rrtConfig)
        result = "path found" if solution else "no path found"
        rrtPRMVisualize(planner, solution, ax=ax, nodeSize=20)
        vis.setupAxes(ax, benchmark.collisionChecker,
                      f"{plannerClass.__name__}: {benchmark.name} (level {benchmark.level})\n"
                      f"{result}, {planner.graph.number_of_nodes()} nodes")

## Statistics

A single run says little about a randomized planner. Therefore every planner is run 20 times on every benchmark (with fixed seeds, so the table is reproducible). The number of line checks is taken from `IPPerfMonitor` (see IP-4-0).

In [ ]:
import pandas as pd
from IPPerfMonitor import IPPerfMonitor

def runStatistics(plannerClass, benchmark, config, runs=20):
    rows = []
    for run in range(runs):
        random.seed(run)
        IPPerfMonitor.clearData()
        planner = plannerClass(benchmark.collisionChecker)
        solution = planner.planPath(benchmark.startList, benchmark.goalList, config)
        calls = IPPerfMonitor.dataFrame()["name"]
        rows.append({"benchmark": benchmark.name, "planner": plannerClass.__name__,
                     "success": bool(solution),
                     "nodes": planner.graph.number_of_nodes(),
                     "nodes in edges": len(getattr(planner, "splitNodes", [])),
                     "path length": vis.pathLength(planner.graph, solution) if solution else None,
                     "line checks": (calls == "lineInCollision").sum()})
    return rows

rows = []
for plannerClass in [RRTSimple, RRT, RRTEdges]:
    for benchmark in ts.benchList:
        rows += runStatistics(plannerClass, benchmark, rrtConfig)

stats = pd.DataFrame(rows)
stats.groupby(["benchmark", "planner"], sort=False).agg(
    success_rate=("success", "mean"), nodes=("nodes", "mean"), nodes_in_edges=("nodes in edges", "mean"),
    path_length=("path length", "mean"), line_checks=("line checks", "mean")).round(2)

Observations:

* "MyField" is solved in every run, "Trap" by `RRTSimple` as well. The two bottlenecks are solved only in about two thirds to four fifths of the runs with 100 nodes: a random sample, whose connection to the tree passes the narrow passage, is rare. The RRT has the same difficulty with narrow passages as the PRM.
* The paths of `RRT` (half the way) are shorter, as the tree consists of shorter edges. But in "Trap" it needs about twice as many nodes and sometimes fails: with half steps the tree grows only slowly out of the trap.
* `RRTEdges` finds shorter paths than `RRTSimple` on all benchmarks, about as short as the ones of `RRT` (half the way), but without its disadvantage in "Trap": it is solved in every run with as many line checks as with `RRTSimple`. In "Bottleneck" it is successful more often (90 % instead of 75 % of the runs) with half the line checks. Its number of nodes includes the nodes inserted into edges.
* The paths of all planners are far from optimal (detours, zigzag); in practice they are smoothed afterwards.

# Experiments and Exercises

1. Change *numberOfGeneratedNodes* and *testGoalAfterNumberOfNodes*. What happens with *testGoalAfterNumberOfNodes* = 1? How does it change the number of line checks?
2. Change `extend` in `planRRT`: instead of a fraction use a fixed step size $\epsilon$ (but never beyond $q_{rand}$).
3. **Goal bias**: with a probability $p$ (e.g. 0.1) use $q_g$ as $q_{rand}$. How does this change the statistics on the benchmarks?
4. Compare the number of line checks with the Basic PRM (IP-5-0) on the same benchmarks.
5. `RRTEdges` searches all edges for every new configuration. How does the planning time grow with the number of nodes? How could a k-d tree of the nodes restrict the search to a few edges?
6. Combine the nearest point of the tree (`RRTEdges`) with extending only part of the way (`RRT`).
7. (Advanced) **RRT-Connect**: grow a second tree from the goal and try to connect both trees.